In [ ]:
## Day 14 - Data Preprocessing & Feature Scaling

What I learned today:
- Fare=512 dominated other features → unfair model
- MinMaxScaler converts all features to 0-1 range
- fit_transform on training data → transform only on test data
- KNeighbors: 69.82% → 80.89% (+11%) after scaling
- SVC: 67.46% → 80.89% (+13%) after scaling
- Random Forest not affected → uses yes/no questions not distances
- Always scale data for distance-based models!

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

print("Day 14 - Data Preprocessing!")

Day 14 - Data Preprocessing!


In [4]:
df = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")

df["Age"] = df["Age"].fillna(df["Age"].mean())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df = df.drop("Cabin", axis=1)
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})

X = df[["Pclass", "Sex", "Age", "Fare", "SibSp", "Parch"]]
y = df["Survived"]

print("Data loaded and cleaned!")
print()
print("Feature ranges BEFORE scaling:")
print(X.describe().loc[["min", "max"]])

Data loaded and cleaned!

Feature ranges BEFORE scaling:
     Pclass  Sex    Age      Fare  SibSp  Parch
min     1.0  0.0   0.42    0.0000    0.0    0.0
max     3.0  1.0  80.00  512.3292    8.0    6.0


In [5]:
# Create the scaler
scaler = MinMaxScaler()

# Fit and transform training data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Scale the data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Check ranges after scaling
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
print("Feature ranges AFTER scaling:")
print(X_train_scaled_df.describe().loc[["min", "max"]])

Feature ranges AFTER scaling:
     Pclass  Sex  Age  Fare  SibSp  Parch
min     0.0  0.0  0.0   0.0    0.0    0.0
max     1.0  1.0  1.0   1.0    1.0    1.0


In [6]:
from sklearn.model_selection import cross_val_score

models_to_test = {
    "KNeighbors": KNeighborsClassifier(),
    "SVC":        SVC(random_state=42)
}

print(f"{'Model':15} {'Before':>10} {'After':>10} {'Improvement':>12}")
print("=" * 50)

for name, model in models_to_test.items():
    # Before scaling
    before = cross_val_score(model, X, y, cv=5).mean()
    
    # After scaling
    after = cross_val_score(model, X_train_scaled, y_train, cv=5).mean()
    
    improvement = (after - before) * 100
    print(f"{name:15} {before*100:>9.2f}% {after*100:>9.2f}% {improvement:>+11.2f}%")

Model               Before      After  Improvement
KNeighbors          69.82%     80.89%      +11.08%
SVC                 67.46%     80.89%      +13.43%
